# 1. Frame the Problem


1. Binary Classification

The primary task of this model is Binary Classification. Given a set of player statistics, the model predicts will classify a player as either an AllStar (1) or not (0). Because the number of AllStar roster spots is fixed each season, this project utilizes a ranking approach, where the model ranks all active players by their predicited probability and then selects the top K players (depending on how many All stars there were that season) as AllStars.

2. Real World Practical Purpose

NBA All Star selections significantly impact player contract incentives, Hall of Fame elgibility, trade value, and overall popularity. However, the selection process involves fan, media, and player voting which can often be influenced by human bias. Some of these biases include :

* Players in larger markets like LA receive more visibility
* Legacy bias: Older stars may receive more votes than younger players
* Injury Replacement players are chosen by the commisioner and may not allign with statistical output

The purpose of this model is to provide an objective benchmark for All Star caliber performance. By training on over 40 years of data, the model can identify which player statistically earned a spot on the team.


# 2. Get the Data

Dataset downloaded from Kaggle.
- [NBA Data](https://www.kaggle.com/datasets/sumitrodatta/nba-aba-baa-stats)

These csv files contain personal statistics for every player from the beginning of the NBA tracking data to the present. The user who provides these datasets claims to have scraped it from BasketballReference and compiled them together into multiple different files.

We will be using 3 of these files : Player Totals.csv, Advanced.csv, and All-Star Selections.csv

###  Import the Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumitrodatta/nba-aba-baa-stats")

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Show all columns in the DataFrame
pd.set_option('display.max_columns', None)

In [ ]:
# Importing Datasets from Kaggle directly
df = pd.read_csv("nbaData/Player Totals.csv")
all_star = pd.read_csv("nbaData/All-Star Selections.csv")
advanced_stats = pd.read_csv("nbaData/Advanced.csv")

# 3. Exploring the Data

In [ ]:
df.head()

All Star Dataset Lists all the players that were classified as AllStars from each NBA season.

We will merge this dataset with our stats dataset later and have this be our target variable.

In [ ]:
all_star.head()

In [ ]:
advanced_stats.head()

# 4. Preparing the Data

### Look for Data that Needs to Be Dropped / Cleaned

In [ ]:
# Drop Rows with NaN values
print(f"Number of rows before dropping NaN values: {len(df)}")
df.dropna(inplace=True)

# Drop Columns with NaN values
df.dropna(axis=1, inplace=True, how = "all")
print(f"Number of rows after dropping NaN values: {len(df)}")

#### Drop duplicate players if they were traded midseason.
In this dataset, if a player is traded their original team stats, their new team stats, and their overall stats are saved. We only want their overall season stats.

In [ ]:
display(df.query('team == "2TM"').head())

In [ ]:
# Drop duplicates if a player was traded midseason. We only keep total season stats.

df = (
    df
    .sort_values(by=['season','player_id', 'g'], ascending=[False, True, False]) #puts highest row up top, which would be 2tm or 3tm or whatever
    .drop_duplicates(subset=['player_id', 'season'], keep='first') # Drop duplicates
)

advanced_stats = (
    advanced_stats
    .sort_values(by=['season','player_id', 'g'], ascending=[False, True, False]) #puts highest row up top, which would be 2tm or 3tm or whatever
    .drop_duplicates(subset=['player_id', 'season'], keep='first') # Drop duplicates
)



## Combine or Add Data


Some stats were not being tracked in the infancy of the league, such as rebounds, steals, blocks and 3 pointers since the line did not exist until 1979. Most importantly, AllStar selections. We will keep data from 1979 and onward.

In [ ]:

all_star = all_star[all_star['season'] >= 1979]  # All-Star data starts from 1973, but 3pt added 1979
# We will also limit the advanced stats dataset to after 1979 to match the All-Star dataset and because the game changed significantly after the 3pt line was added.
advanced_stats = advanced_stats[advanced_stats['season'] >= 1979]


#### Adding All Star selection column at end to predict.
Combining the All Star Selection dataset with our main dataset here.
A 1 indicates an All Star selection, while a 0 indicates they were not selected. The number of all star selections per year varies due to injuries, but usually is around 24-28.

In [ ]:
# Adding column to indicate if player was an All-Str that season. 1 indicates All-Star selection, 0 indicates no selection.
from unicodedata import name


df['AllStar'] = np.where(df[['player_id', 'season']].apply(tuple, axis=1).isin(all_star[['player_id', 'season']].apply(tuple, axis=1)), 1, 0)

# Showw all All-Stars from 2020 season to verify the new column is correct.
display(
    df
    .query("season == 2020 and AllStar == 1")
)

# 5. Training the Model
#### Random Forest
We will build the model on all data prior to 2026 and attempt to predict the allstars for 2026

In [ ]:
features = df.drop(columns = ['season', 'lg', 'player_id', 'player', 'team', 'pos', 'AllStar']).columns

train_df = df[(df['season'] <= 2025) & (df['season'] >= 1980)]
X_train = train_df[features]
y_train = train_df['AllStar']

test_2026_df = df[df['season'] == 2026]
X_test_2026 = test_2026_df[features]
y_test_2026 = test_2026_df['AllStar']

We chose to use A Random Forest Classifier firstly because we have an unbalanced dataset, where only about 24-28 players out of over 400 are chosen as allstars. This model tends to be less prone to overfitting compared to single decision trees. It can also capture the interactions between all the different types of stats we have that linear models cannot.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_first = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 5,
    random_state=42,
)

model_first.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import f1_score
probs = model_first.predict_proba(X_test_2026)[:, 1]

# Attach to results for 2026
results_2026 = test_2026_df[['player', 'AllStar']].copy()
results_2026['Prob'] = probs

# Sort and pick the Top 24
results_2026 = results_2026.sort_values(by='Prob', ascending=False)
results_2026['Predicted_AllStar'] = 0

# Count how many actual All-Stars there are in 2026, and that is how many AllStars our model will predict. This way our predictions are directly comparable to the actuals, since we know there are 24 All-Stars each season before replacement.
actual_all_stars_2026_count = results_2026['AllStar'].sum()

results_2026.iloc[:actual_all_stars_2026_count, results_2026.columns.get_loc('Predicted_AllStar')] = 1

f1 = f1_score(results_2026['AllStar'], results_2026['Predicted_AllStar'])
print(f"F1 Score for 2026 predictions: {f1:.4f}")



# 6. Fine Tuning the Model

### Added Features to Improve Model

New Feature Added (VORP, Usage Percentage, and Win Shares per 48 minutes)

We are combining some of our data here by adding some advanced stats that we deem important. Some other stats are important as well, but they are calculated with basic stats so we will let the model do its work.

In [ ]:

df = df.merge(advanced_stats[['player_id', 'season', 'vorp', 'usg_percent', 'ws_48']],
              on=['player_id', 'season'],
              how='left')

# fill NaNs with 0 for the ML model
df[['vorp', 'usg_percent', 'ws_48']] = df[['vorp', 'usg_percent', 'ws_48']].fillna(0)
df.head()

New Feature Added (Past allStar appearances amount)

Implemented a new column that adds up how many past AllStar appearances a player has, since sometimes players with worse stats still receive votes just because they have become a household name.


In [ ]:
# 1. Sort by player and season (Ascending)
df.sort_values(['player_id', 'season'], ascending=True, inplace=True)

# 2. Create the legacy column
# We group by player, take the AllStar column, and use:
# cumsum() to total All-Star games up to that point
# shift(1) to move it down one row so we don't count the current year
# fillna(0) so that rookies get a 0 instead of a null value
df['as_legacy'] = (
    df.groupby('player_id')['AllStar']
    .transform(lambda x: x.shift(1).fillna(0).cumsum())
    # make an integer instead of a float since it is a count of All-Star appearances
    .astype(int)
)

# 3. Sort it back to previous order
df.sort_values(['season', 'player_id'], ascending=[False, True], inplace=True)
df.head(10)

### Grid Search to Find the Best Parameters

In [ ]:
features = df.drop(columns = ['season', 'lg', 'player_id', 'player', 'team', 'pos', 'AllStar']).columns
train_df = df[(df['season'] <= 2025) & (df['season'] >= 1980)]
X_train = train_df[features]
y_train = train_df['AllStar']

test__df = df[df['season'] == 2026]
X_test = test__df[features]
y_test = test__df['AllStar']

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 10, 12, 15],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

# Initialize the model
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# Setup the Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=2
)

# Run the search
grid_search.fit(X_train, y_train)

# print the best parameters
print("-" * 30)
print("BEST PARAMETERS FOUND:")
print(grid_search.best_params_)
print("-" * 30)

#Save the best performing model to use for predictions
best_model = grid_search.best_estimator_

In [ ]:
best_model = RandomForestClassifier(
    n_estimators = 200,
    max_depth = 12,
    max_features = 'log2',
    min_samples_split = 10,
    random_state=42,
)
best_model.fit(X_train, y_train)


# 7. Analyzing Finalized Model

Here we see the differences in what features the model emphasizes when selecting the best candidates for AllStars. In our first model, before optimizing paramaters or adding our new features, it prioritized points per game.

After FineTuning and adding advanced stats, it now emphasizes VORP, which stands for Value Over Replacement Player. It is used to evaluate overall value and show how much a player improves a team.

In [ ]:

importances = pd.Series(model_first.feature_importances_, index=features.drop(['vorp', 'usg_percent', 'ws_48', 'as_legacy']))
importances.sort_values().plot(kind='barh', title='What makes an All-Star in first model?')

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=features)
importances.sort_values().plot(kind='barh', title='What makes an All-Star in finalized model?')

## Evaluating Results for 2026 Season

In [ ]:
# Get probabilities
probs = best_model.predict_proba(X_test)[:, 1]

# Attach to results
results = test__df[['player', 'AllStar']].copy()
results['Prob'] = probs

# Sort and pick the Top 24
results_2026 = results.sort_values(by='Prob', ascending=False)
results_2026['Predicted_AllStar'] = 0

# Count how many actual All-Stars there are in 2026, and that is how many AllStars our model will predict. This way our predictions are directly comparable to the actuals, since we know there are 24 All-Stars each season before replacement.
actual_all_stars_count = results_2026['AllStar'].sum()

results_2026.iloc[:actual_all_stars_2026_count, results_2026.columns.get_loc('Predicted_AllStar')] = 1




In [ ]:
# Show accuracy, precision, recall, and F1 score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(results_2026['AllStar'], results_2026['Predicted_AllStar'])
precision = precision_score(results_2026['AllStar'], results_2026['Predicted_AllStar'])
recall = recall_score(results_2026['AllStar'], results_2026['Predicted_AllStar'])
f1 = f1_score(results_2026['AllStar'], results_2026['Predicted_AllStar'])

print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")

Here we display the players that the model predicted to be AllStars that season. We choose the same amount of AllStars that were selected the year we are looking at.

The amount should be 24, but injuries and replacements usually increase the amount by a couple.

In [ ]:
print(results_2026.head(actual_all_stars_2026_count)[['player', 'Prob', 'AllStar']])

In [ ]:
# Display Actual All-Stars that the model missed
print("ACTUAL ALL-STARS THE MODEL MISSED (RECALL ERRORS) ")
missed = results_2026[(results_2026['AllStar'] == 1) & (results_2026['Predicted_AllStar'] == 0)]
print(missed[['player', 'Prob']])

# Display Players the model picked who WEREN'T All-Stars (Precision Errors)
print("\nMODEL PICKS WHO WERE NOT ACTUAL ALL-STARS ")
false_positives = results_2026[(results_2026['AllStar'] == 0) & (results_2026['Predicted_AllStar'] == 1)]
print(false_positives[['player', 'Prob']])

# 8. Save the Model

Save the final dataset with predictions for future analysis

In [ ]:

player_stats_cleaned = df
player_stats_cleaned.to_csv("player_stats_with_predictions.csv", index=False)

Save the best model

In [ ]:
import joblib

joblib.dump(best_model, 'best_all_star_predictor.pkl')

# 9. Future Improvements

After evaluation, we see that the model only correctly selected 22 out of 27 All Star players for the 2026 season. Some information that we believe can improve the model is the following:
* Team Success: Sometimes players are on such great teams that their stats take a hit. However, these players are still regularly recognized, as most 1st seeds usually see at least 2 all star selections
* Market Size and Nationally televised games: Adding featrues that account for team market size or amount of nationally televised games could help the model account for players that have a higher visibilty from the public
* Ensemble Stacking: Instead of only Random Forest, we could implement a Stacking Classifier that would train multiple models and combine thier predictions for higher accuracy
* Conference Constraints: Adding a limit to the amount of players from each conference ( theretically would be 12 for the west and 12 for the east), would help improve accuracy


After working with these models, we still believe it would be nearly impossible to create a perfect model as there are many more variables that will always impact how these AllStars are selected. From player storylines to an influx of star players at a certain position, there are always years where certain players make the cut or when players are snubbed that do not make sense to most fans. However, we can always work to improve these models as we gain access to new information and technology.